# Day 22 · 幻觉评测与缓解

**配套讲义**: [`days/day-22.md`](../days/day-22.md) ｜ **本地可跑，不需要 GPU**

用 POPE 式诱导问题测出幻觉率，**同时测漏答率**，再实现「不确定就说不确定」的缓解策略并做 A/B 对比。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w4.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 看 probe 是怎么造的

In [ ]:
import sys; sys.path.insert(0, "..")
from src.eval.hallucination import PLAUSIBLE_OBJECTS, build_probes, MITIGATION_STRATEGIES

print("可混淆物体表（选得越像，测试越有效）:")
for k, v in list(PLAUSIBLE_OBJECTS.items())[:3]:
    print(f"  {k}: {v}")

print("\n四种缓解策略:")
for i, s in enumerate(MITIGATION_STRATEGIES, 1):
    print(f"  {i}. {s if isinstance(s, str) else s.get('name')}")

## 2. 亲手算一遍「只看幻觉率」的陷阱

In [ ]:
# 两个假想模型，在 200 条 probe 上的表现
models = {
    "模型A（爱瞎猜）": {"存在": (91, 100), "不存在": (23, 100)},   # (答对的, 总数)
    "模型B（全说没有）": {"存在": (0, 100), "不存在": (100, 100)},
}
for name, m in models.items():
    hallu = 1 - m["不存在"][0] / m["不存在"][1]     # 不该有的说有 = 幻觉
    miss  = 1 - m["存在"][0] / m["存在"][1]         # 该有的说没有 = 漏答
    acc   = (m["存在"][0] + m["不存在"][0]) / 200
    print(f"{name:16s} 幻觉率 {hallu:5.1%}  漏答率 {miss:5.1%}  总准确 {acc:5.1%}")
print("\n→ 模型B 的幻觉率是 0%，但漏答率 100%：它其实什么都没在看。")

## 3. 设计你自己的属性类 probe

颜色、材质、尺码标、logo 位置 —— 这些「看起来对但可能错」的属性，才是客服场景的真实幻觉来源。

In [ ]:
attribute_probes = {
    # "问题": "该图真实的答案",
    # 例: "这件上衣的领口是什么形状？": "圆领",
}
print("至少写 5 条，Day 23 的错误分析会用到同类型样本。")

## 验收清单

- [ ] 幻觉率与漏答率**两列一起报**（只报一个的结论一律不采信）
- [ ] 能说清三类幻觉（物体存在性 / 属性 / 关系）的成因差异
- [ ] 4 个缓解策略里至少实测 2 个，并用控制变量方式对比
- [ ] 能解释「降低幻觉率往往抬高漏答率」这个 trade-off

**卡住了？** 回看 [`days/day-22.md`](../days/day-22.md) 第五节「容易踩的坑」。

> **明天**：`days/day-23.md` —— 错误分析：把失败样本自动归类